# E5 · Khả năng mở rộng · Chủ: Người 1 · Phục vụ Chương 4 (mục 4.3)
Thời gian huấn luyện theo kích thước dữ liệu → **T4.2 / F4.2**; tốc độ đọc CSV và Parquet → **T4.3**.

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son&Tai"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

In [ ]:
from aitext import data, models, evaluate
train, _, _ = data.load_splits(spark)
train = data.with_class_weight(train); rows = []
for frac in [0.1, 0.25, 0.5, 1.0]:
    part = train.sample(frac, seed=config.SEED).cache(); n = part.count()
    t0 = time.time(); models.fit(part, defended=True); rows.append({"số dòng": n, "giây": time.time() - t0})
    part.unpersist()
t42 = pd.DataFrame(rows); display(t42); evaluate.save_table(t42, "T4.2_scaling")
fig, ax = plt.subplots(figsize=(6, 3.5)); t42.plot(x="số dòng", y="giây", marker="o", ax=ax, legend=False)
ax.set_title("F4.2 Thời gian huấn luyện theo số dòng"); evaluate.save_figure(fig, "F4.2_scaling"); plt.show()

## T4.3: CSV và Parquet (cần tải lại `train_none.csv`, khoảng 5 phút)

In [ ]:
csv = data.download("train_none"); rows = []
for fmt, load in [("CSV", lambda: data.read_raid_csv(spark, csv)), ("Parquet", lambda: spark.read.parquet(config.SPLIT_TRAIN))]:
    t0 = time.time(); d = load(); n = d.count(); t_count = time.time() - t0
    t0 = time.time(); d.groupBy("model").count().collect(); t_group = time.time() - t0
    rows.append({"định dạng": fmt, "số dòng": n, "count (s)": t_count, "groupBy (s)": t_group})
t43 = pd.DataFrame(rows); display(t43); evaluate.save_table(t43, "T4.3_csv_vs_parquet",
    notes="CSV là toàn bộ train_none; Parquet là clean_train đã lấy mẫu, nên ghi rõ số dòng khi so sánh.")